# Step 4. 산불발생 공간·지형·접근성 최종 EDA

이 노트북은 `jsw/강원_재_EDA` 폴더의 마지막 EDA 단계다.

Step 1~3에서 기상, 캐나다 산불지수, 매칭 대조군, 선행 기상, 국지 임계치 후보를 이미 정리했으므로 Step 4에서는 공간·지형·토지피복·접근성과 공간 대조군만 확인한다.

기존 Step 5와 Step 6은 이 EDA 흐름에서 제거한다. 도로·임도·등산로·생활권·산림 내부 같은 인간활동 프록시 요소는 별도 원인 분석이 아니라 Step 4의 접근성·공간층 변수로 흡수한다.

결과 해석은 이 노트북에 작성하지 않는다. 표와 플롯을 함께 검토한 해석, 한계, 다음 반영사항은 `Step4_산불발생_공간지형및대조군_분석_진행예정로그.md`에만 기록한다.

In [1]:
from pathlib import Path
import runpy
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
candidates = [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]
REPO_ROOT = next(
    (path for path in candidates if (path / "jsw/강원_재_EDA").exists()),
    Path(r"D:/farm-system-public-02"),
)
MODULE_DIR = REPO_ROOT / "jsw/강원_재_EDA"
OUT_DIR = MODULE_DIR / "outputs/Step4"
TABLE_DIR = OUT_DIR / "tables"
PLOT_DIR = OUT_DIR / "plots"

## S4-01 : 공간 원천·CRS·geometry 품질 감사

공간 원천 파일 존재 여부, CRS, bounds, geometry validity, 대용량 GPKG 메타데이터와 표본 geometry 유효성을 감사한다.

In [2]:
runpy.run_path(str(MODULE_DIR / "step4_s401_spatial_audit.py"), run_name="__main__")

{'source_rows': 12, 'geometry_rows': 13, 'non_convertible_layers': 0, 'invalid_full_geometry_count': 0, 'plot': 'D:\\farm-system-public-02\\jsw\\강원_재_EDA\\outputs\\Step4\\plots\\S4-01_layer_overlay_quality_map.png'}


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'D:\\farm-system-public-02\\jsw\\강원_재_EDA\\step4_s401_spatial_audit.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, loca

### S4-01 결과표 확인

In [3]:
for name in [
    "S4-01_spatial_source_audit.csv",
    "S4-01_geometry_validity_audit.csv",
    "S4-01_layer_bounds_audit.csv",
]:
    print("\n---", name)
    display(pd.read_csv(TABLE_DIR / name, encoding="utf-8-sig"))


--- S4-01_spatial_source_audit.csv


,key,label,kind,path,exists,bytes,modified,note
0,fire,raw_fire_events,csv_point_wgs84,data\강원도_데이터\강원도_산불발생.csv,True,408260,2026-05-31T04:59:51.900885105,NaN
1,clean_fire,clean_fire_events,csv_point_wgs84,data\학습데이터\산불발생_정제.csv,True,338762,2026-06-11T13:05:44.715772152,reference only; Step 4 spatial audit uses raw ...
2,weather_grid,weather_cell_polygons,csv_wkt_polygon_wgs84,data\강원도_날씨데이터\강원도날씨_격자.csv,True,48182,2026-05-30T10:28:16.485248327,NaN
3,climate_type,climate_topography_type,csv_table,data\강원도_날씨데이터\강원도날씨_기후지형유형_셀분류.csv,True,3111,2026-05-31T10:34:54.482622385,NaN
4,terrain,fire_terrain_features,csv_point_wgs84,data\강원도_데이터\산불_공간데이터\강원도_산불_지형특성계산.csv,True,477820,2026-05-31T05:05:06.908538103,NaN
5,dem,gangwon_dem,raster,data\강원도_데이터\강원도_공간데이터\강원도_DEM_데이터.tif,True,9933156,2026-05-31T05:21:43.481648207,NaN
6,landcover,landcover_fine_gpkg,gpkg,data\강원도_데이터\강원도_공간데이터\강원도_토지피복도_세분류_병합_1m.gpkg,True,1230585856,2026-05-31T07:09:51.222411394,large file; full metadata plus sample geometry...
7,roads,merged_roads_gpkg,gpkg,data\강원도_데이터\강원도_공간데이터\강원도_병합_도로.gpkg,True,307200000,2026-05-31T07:43:12.656644583,large file; full metadata plus sample geometry...
8,trails,hiking_trails,csv_wkt_line_wgs84,data\강원도_데이터\강원도_공간데이터\강원도_등산로.csv,True,20976258,2026-05-31T05:06:41.769750834,NaN
9,forest_roads,forest_roads,csv_wkt_line_wgs84,data\강원도_데이터\강원도_공간데이터\강원도_임도망도.csv,True,20156788,2026-05-31T05:13:49.485568762,NaN



--- S4-01_geometry_validity_audit.csv


,key,label,audit_scope,feature_count,crs,target_crs_convertible,geometry_types,null_geometry_count,empty_geometry_count,invalid_geometry_count,minx,miny,maxx,maxy,width,height,resolution_x,resolution_y,nodata
0,fire,raw_fire_events,full,3405.0,EPSG:4326,True,Point,0.0,0.0,0.0,127.144220,3.706870e+01,1.293513e+02,3.858611e+01,NaN,NaN,NaN,NaN,NaN
1,clean_fire,clean_fire_events,full,1558.0,EPSG:4326,True,Point,0.0,0.0,0.0,127.160133,3.706870e+01,1.293513e+02,3.858611e+01,NaN,NaN,NaN,NaN,NaN
2,weather_grid,weather_cell_polygons,full,92.0,EPSG:4326,True,"MultiPolygon,Polygon",0.0,0.0,0.0,127.095000,3.702780e+01,1.293657e+02,3.861180e+01,NaN,NaN,NaN,NaN,NaN
3,terrain,fire_terrain_features,full,3405.0,EPSG:4326,True,Point,0.0,0.0,0.0,127.144220,3.706870e+01,1.293513e+02,3.858611e+01,NaN,NaN,NaN,NaN,NaN
4,dem,gangwon_dem,raster_metadata,NaN,"PROJCS[""Transverse Mercator"",GEOGCS[""GRS80 ELL...",True,Raster,NaN,NaN,NaN,208276.000000,4.935230e+05,4.101460e+05,6.688430e+05,2243.0,1948.0,90.0,90.0,-9999.0
5,landcover,landcover_fine_gpkg,full_metadata,41119.0,"PROJCS[""PCS_ITRF2000_TM"",GEOGCS[""ITRF2000"",DAT...",True,MultiPolygon,NaN,NaN,NaN,217524.670000,4.930995e+05,4.110727e+05,6.427261e+05,NaN,NaN,NaN,NaN,NaN
6,landcover,landcover_fine_gpkg,first_5000_sample,5000.0,"PROJCS[""PCS_ITRF2000_TM"",GEOGCS[""ITRF2000"",DAT...",True,MultiPolygon,0.0,0.0,0.0,241720.540000,5.530424e+05,2.882790e+05,6.004719e+05,NaN,NaN,NaN,NaN,NaN
7,roads,merged_roads_gpkg,full_metadata,91862.0,EPSG:5179,True,Unknown,NaN,NaN,NaN,965866.352115,1.892516e+06,1.166322e+06,2.069685e+06,NaN,NaN,NaN,NaN,NaN
8,roads,merged_roads_gpkg,first_5000_sample,5000.0,EPSG:5179,True,"MultiPolygon,Polygon",0.0,0.0,0.0,973716.312804,1.892516e+06,1.166322e+06,2.042086e+06,NaN,NaN,NaN,NaN,NaN
9,trails,hiking_trails,full,4523.0,EPSG:4326,True,"LineString,MultiLineString",0.0,0.0,0.0,127.175942,3.702799e+01,1.293549e+02,3.852591e+01,NaN,NaN,NaN,NaN,NaN



--- S4-01_layer_bounds_audit.csv


,key,label,audit_scope,crs,target_crs_convertible,minx,miny,maxx,maxy
0,fire,raw_fire_events,full,EPSG:4326,True,127.144220,3.706870e+01,1.293513e+02,3.858611e+01
1,clean_fire,clean_fire_events,full,EPSG:4326,True,127.160133,3.706870e+01,1.293513e+02,3.858611e+01
2,weather_grid,weather_cell_polygons,full,EPSG:4326,True,127.095000,3.702780e+01,1.293657e+02,3.861180e+01
3,terrain,fire_terrain_features,full,EPSG:4326,True,127.144220,3.706870e+01,1.293513e+02,3.858611e+01
4,dem,gangwon_dem,raster_metadata,"PROJCS[""Transverse Mercator"",GEOGCS[""GRS80 ELL...",True,208276.000000,4.935230e+05,4.101460e+05,6.688430e+05
5,landcover,landcover_fine_gpkg,full_metadata,"PROJCS[""PCS_ITRF2000_TM"",GEOGCS[""ITRF2000"",DAT...",True,217524.670000,4.930995e+05,4.110727e+05,6.427261e+05
6,landcover,landcover_fine_gpkg,first_5000_sample,"PROJCS[""PCS_ITRF2000_TM"",GEOGCS[""ITRF2000"",DAT...",True,241720.540000,5.530424e+05,2.882790e+05,6.004719e+05
7,roads,merged_roads_gpkg,full_metadata,EPSG:5179,True,965866.352115,1.892516e+06,1.166322e+06,2.069685e+06
8,roads,merged_roads_gpkg,first_5000_sample,EPSG:5179,True,973716.312804,1.892516e+06,1.166322e+06,2.042086e+06
9,trails,hiking_trails,full,EPSG:4326,True,127.175942,3.702799e+01,1.293549e+02,3.852591e+01


## S4-02 : 발생지 토지피복·지형 변수 결합

산불 발생지에 토지피복 분류와 지형 특성 데이터를 결합하고, 감사 및 기술통계를 생성합니다.

In [4]:
runpy.run_path(str(MODULE_DIR / "step4_s402_landcover_terrain.py"), run_name="__main__")

--- S4-02: 발생지 토지피복·지형 변수 결합 시작 ---
데이터 로딩 중...


정제 산불 행 수: 1558
지형특성 행 수: 3405
토지피복 폴리곤 수: 41119
지형특성 결합 후 행 수: 1558


토지피복도 공간조인 실행 중...


최종 처리 후 행 수: 1558

--- 공간조인 감사 결과 ---
                   metric  value   ratio_pct
0         모집단 행 수 (정제 산불)   1558  100.000000
1        공간조인 전 임시 매칭 행 수   1569  100.706033
2       다중 매칭(중복) 발생 사건 수     11    0.706033
3  토지피복 공간조인 실패(미매칭) 사건 수    163   10.462131
4   토지피복 공간조인 성공(매칭) 사건 수   1395   89.537869
5             최종 처리 후 행 수   1558  100.000000

--- 번지유형과 실제 토지피복 교차표 ---
           산림 아님(n)   산림 아님(%)  산림지역(n)    산림지역(%)  Total(n)
addr_type                                                   
일반번지           1054  94.275492       64   5.724508      1118
임야번지(산)         246  55.909091      194  44.090909       440
Total          1300  83.440308      258  16.559692      1558

--- 지형 변수 결측 감사 ---
고도(m)            0
경사도(도)           0
TPI(지형위치지수)      0
TWI(지형다습지수)    115
dtype: int64

--- 기후지형유형별 지형 변수 요약표 ---
       variable  기후지형유형  count          mean           std        min  \
0         고도(m)  고지·산간형  223.0  5.331049e+02  1.794650e+02  93.284540   
1         고도(m)  영동 해안형  563.0  

S4-02_landcover_composition.png 생성 완료


S4-02_terrain_ecdf_by_type_elevation.png 생성 완료
S4-02_terrain_ecdf_by_type_slope.png 생성 완료


S4-02_terrain_ecdf_by_type_tpi.png 생성 완료
S4-02_terrain_ecdf_by_type_twi.png 생성 완료
--- S4-02: 발생지 토지피복·지형 변수 결합 완료 ---


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'D:\\farm-system-public-02\\jsw\\강원_재_EDA\\step4_s402_landcover_terrain.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, 

### S4-02 결과 확인

In [5]:
print("\n--- 공간조인 감사 결과 ---")
display(pd.read_csv(TABLE_DIR / "S4-02_fire_landcover_join_audit.csv", encoding="utf-8-sig"))

print("\n--- 번지유형과 실제 토지피복 교차표 ---")
display(pd.read_csv(TABLE_DIR / "S4-02_address_forest_landcover_crosstab.csv", index_col=0, encoding="utf-8-sig"))

print("\n--- 기후지형유형별 지형 변수 요약표 ---")
display(pd.read_csv(TABLE_DIR / "S4-02_fire_terrain_summary.csv", encoding="utf-8-sig"))


--- 공간조인 감사 결과 ---


,metric,value,ratio_pct
0,모집단 행 수 (정제 산불),1558,100.000000
1,공간조인 전 임시 매칭 행 수,1569,100.706033
2,다중 매칭(중복) 발생 사건 수,11,0.706033
3,토지피복 공간조인 실패(미매칭) 사건 수,163,10.462131
4,토지피복 공간조인 성공(매칭) 사건 수,1395,89.537869
5,최종 처리 후 행 수,1558,100.000000



--- 번지유형과 실제 토지피복 교차표 ---


,산림 아님(n),산림 아님(%),산림지역(n),산림지역(%),Total(n)
addr_type,,,,,
일반번지,1054,94.275492,64,5.724508,1118
임야번지(산),246,55.909091,194,44.090909,440
Total,1300,83.440308,258,16.559692,1558



--- 기후지형유형별 지형 변수 요약표 ---


,variable,기후지형유형,count,mean,std,min,25%,50%,75%,max
0,고도(m),고지·산간형,223.0,5.331049e+02,1.794650e+02,93.284540,418.843290,550.386350,658.588020,9.891533e+02
1,고도(m),영동 해안형,563.0,7.004671e+01,8.460261e+01,1.306317,15.526400,44.794773,86.489360,5.426060e+02
2,고도(m),영서 내륙형,772.0,2.820546e+02,1.575908e+02,52.604774,167.229502,239.888285,370.246090,1.163781e+03
3,경사도(도),고지·산간형,223.0,9.115672e+00,6.440241e+00,0.111244,3.869198,8.034257,12.927752,2.992135e+01
4,경사도(도),영동 해안형,563.0,4.739330e+00,4.114957e+00,0.053691,1.731844,3.939325,6.293261,3.236751e+01
5,경사도(도),영서 내륙형,772.0,7.491882e+00,5.670407e+00,0.080065,3.108447,6.246531,10.495640,2.996693e+01
6,TPI(지형위치지수),고지·산간형,223.0,-3.474520e+00,5.736750e+00,-19.685455,-7.200684,-2.938171,-0.365997,1.832538e+01
7,TPI(지형위치지수),영동 해안형,563.0,-1.551318e+00,3.896519e+00,-15.957672,-2.994246,-0.996704,0.078687,1.619081e+01
8,TPI(지형위치지수),영서 내륙형,772.0,-1.772430e+00,5.197813e+00,-27.303452,-3.903320,-1.575827,-0.080238,3.567560e+01
9,TWI(지형다습지수),고지·산간형,219.0,1.250841e+08,4.321624e+08,1.360585,3.522593,4.711554,6.510787,1.611377e+09


## S4-03 : 접근성·공간층 변수 생성

주요 인프라 최단거리 변수의 정합성을 검증하고, 이를 토대로 WUI/산림 접근권/산림 내부 공간층을 할당하며 민감도 분석을 수행합니다.

In [6]:
runpy.run_path(str(MODULE_DIR / "step4_s403_accessibility.py"), run_name="__main__")

--- S4-03: 접근성·공간층 변수 생성 시작 ---
로드된 산불 공간 피처 행 수: 1558
도로 및 등산로/임도 최단거리 정합성 샘플 검증 진행 중...


샘플 20건 최단거리 교차 검증 결과:
  [F_003556] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007615] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007616] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007617] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007618] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007619] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007620] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007621] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007622] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007623] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007624] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007625] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007626] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007627] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007628] 도로 오차: 0.0000m | 임도 오차: 0.0000m | 등산로 오차: 0.0000m
  [F_007629] 도로 오차: 0.0000m | 임도 

S4-03_accessibility_ecdf.png 생성 완료


S4-03_spatial_layer_map.png 생성 완료
--- S4-03: 접근성·공간층 변수 생성 완료 ---


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'D:\\farm-system-public-02\\jsw\\강원_재_EDA\\step4_s403_accessibility.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, loca

### S4-03 결과 확인

In [7]:
print("\n--- 임계치별 공간층 구성비 민감도 분석표 ---")
display(pd.read_csv(TABLE_DIR / "S4-03_access_threshold_sensitivity.csv", encoding="utf-8-sig"))

print("\n--- 기후지형유형별 접근성 변수 요약표 ---")
display(pd.read_csv(TABLE_DIR / "S4-03_accessibility_distance_summary.csv", encoding="utf-8-sig"))


--- 임계치별 공간층 구성비 민감도 분석표 ---


,임계치_m,생활권-WUI(n),생활권-WUI(%),산림 접근권(n),산림 접근권(%),산림 내부(n),산림 내부(%),Total(n)
0,250,1464,93.966624,26,1.668806,68,4.364570,1558
1,500,1464,93.966624,31,1.989730,63,4.043646,1558
2,1000,1464,93.966624,38,2.439024,56,3.594352,1558



--- 기후지형유형별 접근성 변수 요약표 ---


,variable,기후지형유형,count,mean,std,min,25%,50%,75%,max
0,도로_최단거리_m,고지·산간형,223.0,30.428069,146.576288,0.000000,0.000000,2.290541,17.255557,1840.265315
1,도로_최단거리_m,영동 해안형,563.0,21.187855,72.742159,0.000000,0.000000,3.079896,11.698024,936.566738
2,도로_최단거리_m,영서 내륙형,772.0,61.238639,297.718639,0.000000,0.000000,4.610877,19.072675,4774.742876
3,임도_최단거리_m,고지·산간형,223.0,2377.932024,3140.975814,0.424022,921.017376,1765.700719,2910.438781,27048.663273
4,임도_최단거리_m,영동 해안형,563.0,5056.299855,2936.277785,0.796963,3028.163086,4570.900509,6550.005849,26968.592033
5,임도_최단거리_m,영서 내륙형,772.0,4780.638426,3972.977848,1.132785,1801.281840,3525.646592,6694.701049,18951.273258
6,등산로_최단거리_m,고지·산간형,223.0,1390.346699,1572.273826,1.258764,334.872619,855.056345,1912.874537,8793.677915
7,등산로_최단거리_m,영동 해안형,563.0,1086.507637,958.503604,0.619048,332.624061,825.751613,1560.911314,7188.978270
8,등산로_최단거리_m,영서 내륙형,772.0,2527.003633,2346.781788,0.073597,737.630865,1945.442042,3538.180668,16883.667164
9,시가화_최단거리_m,고지·산간형,223.0,371.656635,2841.410401,0.000000,0.000000,1.615978,14.082315,25759.931131


## S4-04 : 공간 대조군 후보풀 생성

In [8]:
s404_outputs = [
    TABLE_DIR / "S4-04_spatial_control_pool.csv",
    TABLE_DIR / "S4-04_spatial_control_pool_audit.csv",
    TABLE_DIR / "S4-04_control_sampling_balance.csv",
    TABLE_DIR / "S4-04_control_exclusion_audit.csv",
    PLOT_DIR / "S4-04_fire_and_control_pool_map.png",
    PLOT_DIR / "S4-04_control_pool_by_cell_layer.png",
]
FORCE_RERUN_S404 = False
if FORCE_RERUN_S404 or not all(path.exists() for path in s404_outputs):
    runpy.run_path(str(MODULE_DIR / "step4_s404_spatial_controls.py"), run_name="__main__")
else:
    print("S4-04 cached outputs already exist; skip rerun.")
    for path in s404_outputs:
        print(path.name)

S4-04 cached outputs already exist; skip rerun.
S4-04_spatial_control_pool.csv
S4-04_spatial_control_pool_audit.csv
S4-04_control_sampling_balance.csv
S4-04_control_exclusion_audit.csv
S4-04_fire_and_control_pool_map.png
S4-04_control_pool_by_cell_layer.png


### S4-04 결과 확인

In [9]:
print("\n--- 공간 대조군 후보풀 감사 ---")
display(pd.read_csv(TABLE_DIR / "S4-04_spatial_control_pool_audit.csv", encoding="utf-8-sig"))

print("\n--- 후보 부족 상위 셀·공간층 ---")
balance = pd.read_csv(TABLE_DIR / "S4-04_control_sampling_balance.csv", encoding="utf-8-sig")
display(balance.sort_values("shortage_n", ascending=False).head(20))

print("\n--- 제외 감사 하위 후보 셀 ---")
exclusion = pd.read_csv(TABLE_DIR / "S4-04_control_exclusion_audit.csv", encoding="utf-8-sig")
display(exclusion.sort_values("eligible_candidate_n").head(20))

print("\n--- 공간층별 대조군 수 ---")
controls = pd.read_csv(TABLE_DIR / "S4-04_spatial_control_pool.csv", encoding="utf-8-sig")
display(controls["spatial_layer_500"].value_counts().rename_axis("spatial_layer_500").reset_index(name="control_n"))


--- 공간 대조군 후보풀 감사 ---


,metric,value,note
0,analysis_id,S4-04,NaN
1,seed,20260614,NaN
2,analysis_unit,공간점 1개,NaN
3,population,"정제 산불 발생지 1,558건이 속한 동일 기상셀 내부의 공간 배경",NaN
4,comparison_group,발생지별 동일 기상셀·동일 spatial_layer_500 대조군 최대 3점,NaN
5,period,2020-01-01 12:00:00 ~ 2021-05-25 14:00:00,NaN
6,controls_per_fire_target,3,NaN
7,exclude_fire_buffer_m,500.0,NaN
8,fire_rows_before,1558,NaN
9,fire_unique_ids,1558,NaN



--- 후보 부족 상위 셀·공간층 ---


,기상셀ID,기후지형유형,spatial_layer_500,fire_n,target_control_n,eligible_candidate_n,assigned_control_n,control_per_fire_mean,fires_with_any_control_n,fires_without_control_n,shortage_n
115,YS_0052,영서 내륙형,생활권-WUI,16,48,0,0,0.0,0,16,48
42,YD_0009,영동 해안형,생활권-WUI,13,39,0,0,0.0,0,13,39
125,YS_0062,영서 내륙형,생활권-WUI,9,27,0,0,0.0,0,9,27
94,YS_0026,영서 내륙형,생활권-WUI,8,24,0,0,0.0,0,8,24
85,YS_0017,영서 내륙형,생활권-WUI,7,21,0,0,0.0,0,7,21
93,YS_0026,영서 내륙형,산림 내부,3,9,0,0,0.0,0,3,9
114,YS_0052,영서 내륙형,산림 내부,3,9,0,0,0.0,0,3,9
27,YS_0059,고지·산간형,산림 내부,3,9,0,0,0.0,0,3,9
84,YS_0017,영서 내륙형,산림 내부,2,6,0,0,0.0,0,2,6
60,YD_0021,영동 해안형,생활권-WUI,2,6,0,0,0.0,0,2,6



--- 제외 감사 하위 후보 셀 ---


,기상셀ID,기후지형유형,raw_sampled_n,eligible_candidate_n,max_raw_n,sampling_geometry_part_n,sampling_geometry_area_km2,excluded_landcover_unmatched_n,excluded_is_water_n,excluded_is_urban_core_n,excluded_too_close_to_fire_n,target_생활권-WUI_n,eligible_생활권-WUI_n,target_산림 내부_n,eligible_산림 내부_n,target_산림 접근권_n,eligible_산림 접근권_n
8,YD_0009,영동 해안형,23400,0,23400,2,74.989897,23400,0,0,1941,39.0,0.0,NaN,NaN,NaN,NaN
20,YD_0021,영동 해안형,4200,0,4000,1,42.822110,4200,0,0,147,6.0,0.0,NaN,NaN,NaN,NaN
46,YS_0026,영서 내륙형,19800,0,19800,1,99.106259,19800,0,0,1040,24.0,0.0,9.0,0.0,NaN,NaN
37,YS_0017,영서 내륙형,16200,0,16200,1,94.675900,16200,0,0,947,21.0,0.0,6.0,0.0,NaN,NaN
75,YS_0059,고지·산간형,7200,0,7200,1,249.796846,7200,0,0,82,3.0,0.0,9.0,0.0,NaN,NaN
78,YS_0062,영서 내륙형,18000,0,18000,1,172.228721,18000,0,0,323,27.0,0.0,3.0,0.0,NaN,NaN
68,YS_0052,영서 내륙형,34200,0,34200,1,149.956408,34200,0,0,2875,48.0,0.0,9.0,0.0,NaN,NaN
64,YS_0047,고지·산간형,300,12,4000,4452,145.748918,0,0,0,3,3.0,12.0,NaN,NaN,NaN,NaN
61,YS_0041,고지·산간형,300,17,5400,3645,73.324915,0,0,0,8,9.0,17.0,NaN,NaN,NaN,NaN
71,YS_0055,고지·산간형,300,27,4000,7082,118.867893,0,0,0,2,6.0,27.0,NaN,NaN,NaN,NaN



--- 공간층별 대조군 수 ---


,spatial_layer_500,control_n
0,생활권-WUI,4224
1,산림 내부,147
2,산림 접근권,90


## S4-05 : 발생지 대 공간 대조군 최소 비교

이 단계에서는 대조군이 배정된 발생지 1,487건과 이에 매칭된 공간 대조군 4,461점(1:3 매칭)을 대상으로 지형, 토지피복, 접근성 변수의 분포 차이를 비교합니다.
대조군이 확보되지 않은 71건의 발생지는 별도로 제외하여 민감도 차이를 분석합니다.

In [10]:
s405_outputs = [
    TABLE_DIR / "S4-05_fire_vs_spatial_control_summary.csv",
    TABLE_DIR / "S4-05_effect_size_by_variable.csv",
    TABLE_DIR / "S4-05_stratified_support_audit.csv",
    TABLE_DIR / "S4-05_control_shortage_sensitivity.csv",
    PLOT_DIR / "S4-05_key_variable_ecdf.png",
    PLOT_DIR / "S4-05_effect_size_forest.png",
]
FORCE_RERUN_S405 = False
if FORCE_RERUN_S405 or not all(path.exists() for path in s405_outputs):
    runpy.run_path(str(MODULE_DIR / "step4_s405_minimum_comparison.py"), run_name="__main__")
else:
    print("S4-05 cached outputs already exist; skip rerun.")
    for path in s405_outputs:
        print(path.name)

S4-05 cached outputs already exist; skip rerun.
S4-05_fire_vs_spatial_control_summary.csv
S4-05_effect_size_by_variable.csv
S4-05_stratified_support_audit.csv
S4-05_control_shortage_sensitivity.csv
S4-05_key_variable_ecdf.png
S4-05_effect_size_forest.png


### S4-05 결과 확인

In [11]:
print("\n--- 대조군 미배정 발생지 민감도 분석 (Matched vs All) ---")
display(pd.read_csv(TABLE_DIR / "S4-05_control_shortage_sensitivity.csv", encoding="utf-8-sig"))

print("\n--- 주 분석 지형/접근성 변수별 기술통계 비교 ---")
display(pd.read_csv(TABLE_DIR / "S4-05_fire_vs_spatial_control_summary.csv", encoding="utf-8-sig"))

print("\n--- 주 분석 변수별 통계적 효과크기 및 검정 요약 ---")
display(pd.read_csv(TABLE_DIR / "S4-05_effect_size_by_variable.csv", encoding="utf-8-sig"))


--- 대조군 미배정 발생지 민감도 분석 (Matched vs All) ---


,variable,group,count,mean,std,min,25%,50%,75%,max
0,고도(m),Matched (n=1487),1487,241.362537,208.441271,1.306317,66.283502,186.988310,367.665695,1163.781100
1,고도(m),All (n=1558),1558,241.376650,207.598961,1.306317,67.258790,191.126880,361.278570,1163.781100
2,경사도(도),Matched (n=1487),1487,6.810763,5.561136,0.053691,2.641774,5.205589,9.539705,32.367508
3,경사도(도),All (n=1558),1558,6.729635,5.521186,0.053691,2.603717,5.139028,9.446051,32.367508
4,TPI(지형위치지수),Matched (n=1487),1487,-1.997446,4.978546,-27.303452,-4.202927,-1.562927,-0.058343,35.675600
5,TPI(지형위치지수),All (n=1558),1558,-1.936153,4.894584,-27.303452,-4.042976,-1.483040,-0.051001,35.675600
6,도로_최단거리_m,Matched (n=1487),1487,39.763541,219.648281,0.000000,0.000000,3.255297,14.723436,4774.742876
7,도로_최단거리_m,All (n=1558),1558,42.355873,221.864116,0.000000,0.000000,3.660546,15.834226,4774.742876
8,시가화_최단거리_m,Matched (n=1487),1487,250.287110,1183.913156,0.000000,0.000000,2.593592,15.721277,15883.698546
9,시가화_최단거리_m,All (n=1558),1558,684.211519,2540.237682,0.000000,0.000000,3.181327,22.761672,25759.931131



--- 주 분석 지형/접근성 변수별 기술통계 비교 ---


,spatial_layer,variable,fire_n,fire_mean,fire_std,fire_median,control_n,control_mean,control_std,control_median,difference
0,전체,고도(m),1487,241.362537,208.441271,186.988310,4459,320.116340,259.530872,249.236221,-78.753804
1,전체,경사도(도),1487,6.810763,5.561136,5.205589,4459,10.341006,7.014671,9.343106,-3.530244
2,전체,TPI(지형위치지수),1487,-1.997446,4.978546,-1.562927,4459,-1.252809,6.398711,-1.108002,-0.744637
3,전체,도로_최단거리_m,1487,39.763541,219.648281,3.255297,4461,86.409809,194.833950,46.815369,-46.646268
4,전체,시가화_최단거리_m,1487,250.287110,1183.913156,2.593592,4461,97.102726,232.710531,47.648329,153.184384
5,전체,농업_최단거리_m,1487,304.887513,1245.827363,21.959022,4461,199.630476,401.928704,56.570870,105.257037
6,전체,산림_최단거리_m,1487,270.446232,1133.876869,31.635818,4461,31.648850,83.089915,0.000000,238.797382
7,전체,임도_최단거리_m,1487,4118.694391,2958.065281,3416.268893,4461,3689.706454,3114.371836,3046.162198,428.987937
8,전체,등산로_최단거리_m,1487,1801.010791,1930.133601,1167.663885,4461,1910.337554,1846.280688,1370.663958,-109.326763
9,생활권-WUI,고도(m),1408,231.254668,204.024821,176.424245,4222,311.329169,258.366934,236.058540,-80.074501



--- 주 분석 변수별 통계적 효과크기 및 검정 요약 ---


,spatial_layer,variable,fire_n,control_n,difference_in_means,ci_95_lower,ci_95_upper,welch_p_value,welch_q_value_fdr,mwu_p_value,cohens_d,cliffs_delta
0,전체,고도(m),1487,4459,-78.753804,-91.807559,-65.700049,0.000000e+00,0.000000e+00,6.748234e-24,-0.317878,-0.174297
1,전체,경사도(도),1487,4459,-3.530244,-3.880069,-3.180418,0.000000e+00,0.000000e+00,1.067277e-69,-0.528400,-0.305135
2,전체,TPI(지형위치지수),1487,4459,-0.744637,-1.059880,-0.429394,3.775240e-06,4.853880e-06,4.994161e-08,-0.122577,-0.094261
3,전체,도로_최단거리_m,1487,4461,-46.646268,-59.195701,-34.096836,4.263256e-13,9.592327e-13,0.000000e+00,-0.231699,-0.652937
4,전체,시가화_최단거리_m,1487,4461,153.184384,92.575523,213.793244,7.934120e-07,1.428142e-06,3.552482e-258,0.245004,-0.593007
5,전체,농업_최단거리_m,1487,4461,105.257037,40.797494,169.716579,1.387470e-03,1.560904e-03,2.682847e-19,0.147525,-0.154799
6,전체,산림_최단거리_m,1487,4461,238.797382,181.067661,296.527102,1.110223e-15,3.330669e-15,3.256304e-174,0.417922,0.454999
7,전체,임도_최단거리_m,1487,4461,428.987937,252.961773,605.014101,1.859682e-06,2.789523e-06,6.216338e-10,0.139461,0.106935
8,전체,등산로_최단거리_m,1487,4461,-109.326763,-221.450971,2.797445,5.599165e-02,5.599165e-02,4.641005e-04,-0.058539,-0.060526
9,생활권-WUI,고도(m),1408,4222,-80.074501,-93.282273,-66.866728,0.000000e+00,0.000000e+00,7.823648e-24,-0.325625,-0.178863


## S4-06 : 최종 EDA 공간 요약

이 단계에서는 Step 4 공간/지형/접근성 분석 결과를 바탕으로, 최종 요약 보고서에 반영할 메시지, 후속 모델링 이관 변수(Handoff), 해석의 한계점을 요약합니다.

In [12]:
s406_outputs = [
    TABLE_DIR / "S4-06_final_spatial_eda_summary.csv",
    TABLE_DIR / "S4-06_spatial_variable_handoff.csv",
    TABLE_DIR / "S4-06_interpretation_limits.csv",
    PLOT_DIR / "S4-06_final_spatial_summary_plot.png",
]
FORCE_RERUN_S406 = False
if FORCE_RERUN_S406 or not all(path.exists() for path in s406_outputs):
    runpy.run_path(str(MODULE_DIR / "step4_s406_final_summary.py"), run_name="__main__")
else:
    print("S4-06 cached outputs already exist; skip rerun.")
    for path in s406_outputs:
        print(path.name)

S4-06 cached outputs already exist; skip rerun.
S4-06_final_spatial_eda_summary.csv
S4-06_spatial_variable_handoff.csv
S4-06_interpretation_limits.csv
S4-06_final_spatial_summary_plot.png


### S4-06 결과 확인

In [13]:
print("\n--- S4-06 최종 공간 EDA 요약 메시지표 ---")
display(pd.read_csv(TABLE_DIR / "S4-06_final_spatial_eda_summary.csv", encoding="utf-8-sig"))

print("\n--- S4-06 후속 모델링 이관 변수 등급 분류표 ---")
display(pd.read_csv(TABLE_DIR / "S4-06_spatial_variable_handoff.csv", encoding="utf-8-sig"))

print("\n--- S4-06 물리 해석의 범주 및 해석 한계점표 ---")
display(pd.read_csv(TABLE_DIR / "S4-06_interpretation_limits.csv", encoding="utf-8-sig"))


--- S4-06 최종 공간 EDA 요약 메시지표 ---


,요약 메시지,근거 표,근거 플롯,해석 가능 범위,해석 금지 문장
0,도로 초근접성 편향: WUI 내 산불은 도로 10m 이내에 70% 집중됨,S4-05_effect_size_by_variable.csv (Cliff's del...,S4-05_key_variable_ecdf.png,동일한 기상 및 공간층 통제 하에서도 발생지가 대조군보다 도로에 극도로 근접하여 발화함,도로변 담뱃재 투기가 산불의 직접적 주원인이라고 인과 단정하는 것
1,인프라 접경지 발원 패턴: WUI 내 산불은 과반이 시가화 및 초지 경계선에 집중되...,"S4-05_landcover_crosstab.csv (시가화지역 39.3%, 초지 ...",NaN,"산불 발원은 숲 내부가 아닌 인프라 접경지 나대지, 마당, 도로변 수풀에서 유래함","산림 면적이 좁아서 산불이 잘 난다거나, 건물 자체가 산불을 유도한다고 해석하는 것"
2,지형의 비선형성 및 층화: 발생지는 전반적으로 고도가 낮고 경사가 완만한 골짜기에 ...,S4-05_effect_size_by_variable.csv (산림 내부 TPI C...,S4-05_effect_size_forest.png,공간층에 따라 고도/경사의 효과 크기가 달라지며 지형적 영향이 비선형적으로 반전됨,특정 고도나 TPI 수치가 산불을 유발하는 직접적 원인이라고 단정하는 것
3,산림 접근권 내 등산로 영향: 임도/등산로 500m 이내 영역에서는 등산로 최단거리...,S4-05_effect_size_by_variable.csv (산림 접근권 등산로 ...,S4-05_effect_size_forest.png,인접 인프라가 한정된 산림 접근권 영역 내부에서는 등산로 인접성이 공간적 취약 지표...,등산객이 100% 고의로 산불을 냈다고 단정하는 것



--- S4-06 후속 모델링 이관 변수 등급 분류표 ---


,변수명,등급,Cliff's delta (WUI),권장 피처 처리 방식,선정/제외 사유
0,도로_최단거리_m,EDA 핵심,-0.725,도로 10m 이내 여부 더미 변수(is_road_ultra_close) 및 선형 거...,WUI 내부에서도 가장 강력하고 압도적인 공간 편향을 보임
1,시가화_최단거리_m,후속 모델링 후보,-0.663,시가화 10m 이내 여부(is_urban_edge) 및 선형 거리,"매우 강한 비선형적 쏠림이 있으나, 도로와 상관관계가 높으므로 다중공선성 확인 필요"
2,산림_최단거리_m,후속 모델링 후보,0.466,산림 내부(0m) vs 산림 외부 경계 거리 분리,발생지가 산림 경계면 외부 190m 부근 WUI 구역에 치우쳐 있어 경계부 정의에 필수적
3,경사도(도),후속 모델링 후보,-0.318,선형 경사도 변수,발생지가 완만한 사면에 쏠리는 경향이 전 공간층에서 통계적으로 유의함
4,고도(m),후속 모델링 후보,-0.179,기후지형유형 권역 층화 결합 피처,"권역별(영동/영서/산간) 고도 분포가 뚜렷이 분리되며, 발생지가 상대적으로 저지대에..."
5,농업_최단거리_m,보고서 보조,-0.168,선형 거리 변수,WUI 내부에서 비교 시 다른 변수들에 비해 상대적으로 편향의 강도가 약함
6,TPI(지형위치지수),후속 모델링 후보,-0.106,산림 내부 vs WUI 층별 부호 반전 인터랙션 피처,"산림 내부에서는 능선부(+), WUI에서는 계곡부(-)로 편향이 반전되어 층별 교차..."
7,임도_최단거리_m,보고서 보조,0.104,선형 거리 변수,"WUI 내부 발화에는 미치는 영향이 작으나, 산림 접근권 층에서는 보조 피처로 유효함"
8,등산로_최단거리_m,후속 모델링 후보,-0.068,산림 접근권 층에서의 인터랙션 피처(is_forest_access * hiking_...,산림 접근권 층 내부(Cliff's delta -0.219)에서 뚜렷한 음수 효과를 보임
9,TWI(지형다습지수),제외/보류,NaN,NaN,수문학적 흐름 누적 연산 한계로 대조군에서 계산 불가하여 주 비교 변수에서 보류



--- S4-06 물리 해석의 범주 및 해석 한계점표 ---


,구분,해석 가능 범위,해석 금지/주의 사항,대안 및 보완책
0,도로/인프라 인접성,동일 기상 및 공간층 하에서 발생지가 대조군보다 도로 및 시가화 구역에 물리적으로 ...,도로변 담뱃재 투기나 차량 화재 등이 산불의 직접 원인이라고 인과를 단정 짓는 것,"물리적 노출 및 발원 취약 지대로만 기술하고, 발화 원인은 소방청 화인 통계 등과 ..."
1,토지피복과 WUI 경계,WUI 내 산불의 85% 이상이 실제 산림 피복 바깥의 시가화/초지 등 인프라 접경...,"산림 면적이 좁은 지역이 산불에 취약하다거나, 건물 자체가 산불을 유도한다고 해석하는 것",인간 활동 반경의 끝이자 산림 연료층의 시작점인 WUI 경계부가 물리적 연료 교차 ...
2,지형 지표 (고도/경사),산불 발생지가 상대적으로 완만한 경사 및 저지대 골짜기 사면에 편향되어 분포함,특정 고도나 경사 수치가 산불을 유발하는 직접적 인자라고 인과적으로 해석하는 것,기후지형유형 권역 분류에 따른 고도 분포의 층화 양상과 바람 경로(골짜기 푄 효과)...
3,대조군 부족 71건,71건의 대조군 부족은 강원도 격자 경계선 및 토지피복 미매칭 품질 한계에 따른 것...,"이 71건의 제외가 주 분석 결과의 통계적 유의성에 왜곡을 초래했거나, 대조군이 없...","선택 편향 검증(민감도 표) 결과를 제시하고, 격자 경계부 토지피복 GIS 데이터 ..."
